# Local Volatility Benchmark: Raw SVI Surface, Dupire Inversion and One-Day Hedging (CLEANING IS DIFFERENT FROM BS NOTEBOOKS!)

This notebook implements the **provisional best-performing LV pipeline** identified in the preliminary repository comparison:

$$
\text{IvyDB cleaning}
\longrightarrow
\text{OTM implied-volatility surface}
\longrightarrow
\text{raw SVI slices}
\longrightarrow
\text{calendar-safe smoothing of total variance}
\longrightarrow
\text{Dupire local volatility}
\longrightarrow
\text{Crank--Nicolson pricing and delta hedging}.
$$

It uses the **preliminary LV cleaning universe**, rather than the later common BS benchmark universe:

- SPX (`secid = 108105`);
- European, standard-size options;
- $7 \leq \mathrm{DTE} \leq 365$;
- $|\log(K/F)| \leq 0.30$;
- $0.05 \leq |\Delta| \leq 0.95$;
- relative bid–ask spread no greater than $25\%$;
- OTM put below the forward and OTM call above the forward;
- near ATM, use the option side with the tighter spread.

The implementation is deliberately transparent. Each stage exposes diagnostics rather than silently hiding derivative clipping or arbitrage repairs.

## Main theoretical references

1. Dupire, *Pricing with a Smile* (1994).
2. Gatheral, *The Volatility Surface: A Practitioner's Guide*, Chapters 1 and 3.
3. Gatheral and Jacquier, *Arbitrage-free SVI volatility surfaces*.
4. Dumas, Fleming and Whaley, *Implied Volatility Functions: Empirical Tests*.
5. The repository experiments discussed in the accompanying project notes:
   `incroyale/volatility_surface`, `arkonique/ssvi`,
   `Theo-Sullivan/Arbitrage-free-interpolation-of-SSVI-slices`,
   and `IceCurrent/local_volatility_model`.

## 1. Imports, file paths and visible configuration

In [26]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import PchipInterpolator, RegularGridInterpolator
from scipy.optimize import least_squares
from scipy.stats import norm
from sklearn.isotonic import IsotonicRegression
from tqdm import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

DATA_DIR = Path(".")

OPTIONS_FILE = DATA_DIR / "one_month_spx_option_price.csv"
FORWARD_FILE = DATA_DIR / "one_month_spx_forward_price.csv"
SPOT_FILE = DATA_DIR / "one_month_spx_security_price.csv"
RATES_FILE = DATA_DIR / "one_month_spx_zero_coupon_yield_curve.csv"

SECID = 108105

# Preliminary LV cleaning
MIN_DTE = 7
MAX_DTE = 365
MAX_ABS_LOG_MONEYNESS = 0.30
MIN_ABS_DELTA = 0.05
MAX_ABS_DELTA = 0.95
MAX_RELATIVE_SPREAD = 0.25
ATM_TIGHTER_SIDE_BAND = 0.05
MIN_POINTS_PER_SLICE = 8

# Surface grid
K_MIN, K_MAX, N_K = -0.30, 0.30, 241
DUPIRE_WT_FLOOR = 1e-7
DUPIRE_G_FLOOR = 1e-6
LOCAL_VOL_MIN = 0.01
LOCAL_VOL_MAX = 1.50

# PDE defaults. Increase NX/NT for final reporting.
PDE_NX = 301
PDE_NT_PER_YEAR = 900
PDE_X_WIDTH = 0.75

# Execution controls
EXAMPLE_DATE = None           # None -> first available date
RUN_FULL_SAME_DAY_PDE = False # True is computationally expensive
RUN_ROLLING_BACKTEST = False  # set True after inspecting same-day diagnostics
MAX_ROLLING_TRANSITIONS = None

## 2. Load the raw IvyDB files

In [27]:
options_raw = pd.read_csv(OPTIONS_FILE, low_memory=False)
forwards_raw = pd.read_csv(FORWARD_FILE, low_memory=False)
spot_raw = pd.read_csv(SPOT_FILE, low_memory=False)
rates_raw = pd.read_csv(RATES_FILE, low_memory=False)

print("Raw rows")
print(" options:", len(options_raw))
print(" forwards:", len(forwards_raw))
print(" spot:", len(spot_raw))
print(" rates:", len(rates_raw))

Raw rows
 options: 326964
 forwards: 1192
 spot: 20
 rates: 220


## 3. Basic option cleaning

OptionMetrics reports `strike_price` in thousandths of an index point, so the economic strike is

$$
K=\frac{\texttt{strike\_price}}{1000}.
$$

The quote midpoint and relative spread are

$$
M=\frac{\mathrm{bid}+\mathrm{ask}}{2},
\qquad
s_{\mathrm{rel}}=\frac{\mathrm{ask}-\mathrm{bid}}{M}.
$$

The filters here are the preliminary LV filters used in the repository comparison.

In [28]:
options = options_raw.copy()

options["date"] = pd.to_datetime(options["date"])
options["exdate"] = pd.to_datetime(options["exdate"])
options["K"] = options["strike_price"] / 1000.0
options["T_days"] = (options["exdate"] - options["date"]).dt.days
options["T"] = options["T_days"] / 365.0

options["bid"] = pd.to_numeric(options["best_bid"], errors="coerce")
options["ask"] = pd.to_numeric(options["best_offer"], errors="coerce")
options["mid"] = 0.5 * (options["bid"] + options["ask"])
options["spread"] = options["ask"] - options["bid"]
options["relative_spread"] = options["spread"] / options["mid"].replace(0, np.nan)

options["iv_market"] = pd.to_numeric(options["impl_volatility"], errors="coerce")
options["delta_market"] = pd.to_numeric(options["delta"], errors="coerce")
options["abs_delta_market"] = options["delta_market"].abs()

options = options[
    (options["secid"] == SECID)
    & (options["exercise_style"] == "E")
    & (options["contract_size"] == 100)
    & (options["cfadj"] == 1)
    & options["cp_flag"].isin(["C", "P"])
    & (options["bid"] >= 0)
    & (options["ask"] > options["bid"])
    & (options["mid"] > 0)
    & (options["T_days"].between(MIN_DTE, MAX_DTE))
    & options["iv_market"].between(1e-4, 5.0)
    & options["abs_delta_market"].between(MIN_ABS_DELTA, MAX_ABS_DELTA)
    & (options["relative_spread"] <= MAX_RELATIVE_SPREAD)
].copy()

print("Rows after basic option filters:", len(options))

Rows after basic option filters: 197334


## 4. Clean and merge spot and forward prices

In [29]:
spot = spot_raw.copy()
spot["date"] = pd.to_datetime(spot["date"])
spot = (
    spot[spot["secid"] == SECID]
    [["date", "close"]]
    .rename(columns={"close": "S"})
    .drop_duplicates("date")
)

forwards = forwards_raw.copy()
forwards["date"] = pd.to_datetime(forwards["date"])
forwards["expiration"] = pd.to_datetime(forwards["expiration"])
forwards = forwards[forwards["secid"] == SECID].copy()
forwards = forwards.rename(
    columns={
        "expiration": "exdate",
        "AMSettlement": "am_settlement",
        "ForwardPrice": "F",
    }
)
forwards = forwards[["date", "exdate", "am_settlement", "F"]].drop_duplicates(
    ["date", "exdate", "am_settlement"]
)

options = options.merge(spot, on="date", how="left")
options = options.merge(
    forwards,
    on=["date", "exdate", "am_settlement"],
    how="left",
    validate="many_to_one",
)

print("Missing spot:", options["S"].isna().mean())
print("Missing supplied forward:", options["F"].isna().mean())

Missing spot: 0.0
Missing supplied forward: 0.0


## 5. Interpolate the zero-coupon curve

For every valuation date, the continuously compounded OptionMetrics zero rate is linearly interpolated in maturity. Rates in the CSV are percentages, hence the division by $100$.

$$
D(T)=e^{-r(T)T}.
$$

The forward then implies the continuously compounded carry

$$
c(T)=r(T)-q(T)=\frac{\log(F(T)/S_0)}{T}.
$$

In [30]:
rates = rates_raw.copy()
rates["date"] = pd.to_datetime(rates["date"])
rates["days"] = pd.to_numeric(rates["days"], errors="coerce")
rates["rate_decimal"] = pd.to_numeric(rates["rate"], errors="coerce") / 100.0
rates = rates.dropna(subset=["date", "days", "rate_decimal"]).sort_values(["date", "days"])

rate_curves = {
    d: (g["days"].to_numpy(float), g["rate_decimal"].to_numpy(float))
    for d, g in rates.groupby("date")
}

def interpolated_rate(date, days):
    curve = rate_curves.get(pd.Timestamp(date))
    if curve is None or not np.isfinite(days):
        return np.nan
    x, y = curve
    return float(np.interp(days, x, y, left=y[0], right=y[-1]))

options["r"] = [
    interpolated_rate(d, n)
    for d, n in zip(options["date"], options["T_days"])
]
options["D"] = np.exp(-options["r"] * options["T"])
options["carry"] = np.log(options["F"] / options["S"]) / options["T"]
options["q"] = options["r"] - options["carry"]
options["k"] = np.log(options["K"] / options["F"])

options = options.dropna(subset=["S", "F", "r", "D", "carry", "k"]).copy()
options = options[options["k"].abs() <= MAX_ABS_LOG_MONEYNESS].copy()

print("Rows after forward/rate/moneyness construction:", len(options))

Rows after forward/rate/moneyness construction: 192031


## 6. Static no-arbitrage bounds and the preliminary common quote universe

For a European option with deterministic rates and carry:

$$
C \in \left[D(F-K)^+,\, DF\right],
\qquad
P \in \left[D(K-F)^+,\, DK\right].
$$

Quotes whose midpoint lies materially outside these bounds are removed. A small tolerance is retained because the daily files are not perfectly synchronous.

In [31]:
tol = 1e-6

call_lb = options["D"] * np.maximum(options["F"] - options["K"], 0.0)
call_ub = options["D"] * options["F"]
put_lb = options["D"] * np.maximum(options["K"] - options["F"], 0.0)
put_ub = options["D"] * options["K"]

lb = np.where(options["cp_flag"].eq("C"), call_lb, put_lb)
ub = np.where(options["cp_flag"].eq("C"), call_ub, put_ub)

options = options[
    (options["mid"] >= lb - tol)
    & (options["mid"] <= ub + tol)
].copy()

options["pair_key"] = (
    options["date"].astype(str)
    + "|"
    + options["exdate"].astype(str)
    + "|"
    + options["am_settlement"].astype(str)
    + "|"
    + options["K"].astype(str)
)

quotes_all = options.sort_values(
    ["date", "exdate", "am_settlement", "K", "cp_flag"]
).reset_index(drop=True)

print("Final preliminary quote universe:", len(quotes_all))
print("Dates:", quotes_all["date"].nunique())
print("Date-expiry slices:", quotes_all.groupby(["date", "exdate", "am_settlement"]).ngroups)

Final preliminary quote universe: 192031
Dates: 20
Date-expiry slices: 809


## 7. Black--Scholes pricing, vega and implied-volatility validation

Prices are written in forward form:

$$
C=D F\left[N(d_1)-e^kN(d_2)\right],
\qquad
P=D F\left[e^kN(-d_2)-N(-d_1)\right],
$$

where

$$
d_1=-\frac{k}{\sqrt w}+\frac{\sqrt w}{2},
\qquad
d_2=d_1-\sqrt w.
$$

In [32]:
def bs_forward_price(cp_flag, F, K, T, D, sigma):
    F, K, T, D, sigma = np.broadcast_arrays(
        np.asarray(F, float),
        np.asarray(K, float),
        np.asarray(T, float),
        np.asarray(D, float),
        np.asarray(sigma, float),
    )
    sqrtT = np.sqrt(np.maximum(T, 1e-16))
    volT = np.maximum(sigma, 1e-12) * sqrtT
    d1 = (np.log(np.maximum(F, 1e-300) / np.maximum(K, 1e-300)) + 0.5 * volT**2) / volT
    d2 = d1 - volT
    call = D * (F * norm.cdf(d1) - K * norm.cdf(d2))
    put = D * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
    return np.where(np.asarray(cp_flag) == "C", call, put)

def bs_spot_delta(cp_flag, F, K, T, D, sigma, carry):
    volT = np.maximum(np.asarray(sigma, float), 1e-12) * np.sqrt(np.maximum(T, 1e-16))
    d1 = (np.log(np.asarray(F) / np.asarray(K)) + 0.5 * volT**2) / volT
    dq = np.exp(-(np.asarray(-carry + 0.0)) * 0.0)  # placeholder; use F/S identity below
    forward_to_spot = np.asarray(F) / np.maximum(np.asarray(F) * np.exp(-np.asarray(carry) * T), 1e-300)
    # D * dF/dS = D * exp(carry*T)
    multiplier = np.asarray(D) * np.exp(np.asarray(carry) * T)
    call_delta = multiplier * norm.cdf(d1)
    put_delta = multiplier * (norm.cdf(d1) - 1.0)
    return np.where(np.asarray(cp_flag) == "C", call_delta, put_delta)

quotes_all["bs_market_iv_price"] = bs_forward_price(
    quotes_all["cp_flag"], quotes_all["F"], quotes_all["K"],
    quotes_all["T"], quotes_all["D"], quotes_all["iv_market"]
)
quotes_all["market_iv_repricing_error"] = quotes_all["bs_market_iv_price"] - quotes_all["mid"]

print("Market-IV repricing MAE:",
      quotes_all["market_iv_repricing_error"].abs().mean())

Market-IV repricing MAE: 0.14517807990320855


## 8. Construct the OTM calibration surface

At each $(K,T)$:

- use the put if $K<F$;
- use the call if $K\geq F$;
- inside $|k|\leq0.05$, use whichever side has the tighter price spread.

The output is one calibration observation per date, expiry and strike. Total implied variance is

$$
w_{\mathrm{mkt}}(k,T)=\sigma_{\mathrm{mkt}}^2(k,T)T.
$$

In [33]:
def build_otm_surface(quotes):
    idx_cols = [
        "date", "exdate", "am_settlement", "K", "T",
        "S", "F", "r", "q", "carry", "D", "k"
    ]

    wide = quotes.pivot_table(
        index=idx_cols,
        columns="cp_flag",
        values=["mid", "bid", "ask", "spread", "relative_spread",
                "iv_market", "delta_market", "optionid"],
        aggfunc="first",
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    use_call = wide["K"] >= wide["F"]
    near = wide["k"].abs() <= ATM_TIGHTER_SIDE_BAND

    call_spread = wide.get("spread_C", pd.Series(np.inf, index=wide.index))
    put_spread = wide.get("spread_P", pd.Series(np.inf, index=wide.index))
    tighter_call = call_spread <= put_spread

    choose_call = np.where(near, tighter_call, use_call)
    have_call = wide.get("iv_market_C", pd.Series(np.nan, index=wide.index)).notna()
    have_put = wide.get("iv_market_P", pd.Series(np.nan, index=wide.index)).notna()
    choose_call = np.where(~have_call & have_put, False, choose_call)
    choose_call = np.where(~have_put & have_call, True, choose_call)

    wide["cp_otm"] = np.where(choose_call, "C", "P")
    for base in ["mid", "bid", "ask", "spread", "relative_spread",
                 "iv_market", "delta_market", "optionid"]:
        c = wide.get(f"{base}_C", pd.Series(np.nan, index=wide.index))
        p = wide.get(f"{base}_P", pd.Series(np.nan, index=wide.index))
        wide[f"{base}_otm"] = np.where(choose_call, c, p)

    wide = wide.dropna(subset=["iv_market_otm", "mid_otm"]).copy()
    wide["w_market"] = wide["iv_market_otm"]**2 * wide["T"]

    # Mild robust weight. Equal weighting remains dominant; the spread only
    # prevents the noisiest accepted quotes from dominating.
    wide["fit_weight"] = 1.0 / np.maximum(wide["relative_spread_otm"], 0.02)
    wide["fit_weight"] /= wide.groupby(["date", "exdate", "am_settlement"])["fit_weight"].transform("mean")
    wide["fit_weight"] = wide["fit_weight"].clip(0.25, 4.0)

    return wide.sort_values(["date", "T", "k"]).reset_index(drop=True)

otm = build_otm_surface(quotes_all)

slice_sizes = otm.groupby(["date", "exdate", "am_settlement"]).size()
valid_slices = slice_sizes[slice_sizes >= MIN_POINTS_PER_SLICE].index
otm = (
    otm.set_index(["date", "exdate", "am_settlement"])
    .loc[valid_slices]
    .reset_index()
)

print("OTM observations:", len(otm))
print("Usable slices:", otm.groupby(["date", "exdate", "am_settlement"]).ngroups)

OTM observations: 102980
Usable slices: 809


## 9. Raw SVI slice calibration

For each maturity,

$$
w(k)=a+b\left\{\rho(k-m)+\sqrt{(k-m)^2+\sigma^2}\right\}.
$$

The fit enforces the elementary parameter restrictions

$$
b\geq0,\qquad |\rho|<1,\qquad \sigma>0,
$$

and uses a transformed intercept so that the global minimum of total variance remains positive:

$$
a+b\sigma\sqrt{1-\rho^2}>0.
$$

Multiple starts and maturity warm starts improve robustness. The fitting objective is weighted least squares in **total variance**, not implied volatility.

In [34]:
def svi_raw_w(k, a, b, rho, m, sigma):
    z = np.asarray(k, float) - m
    return a + b * (rho * z + np.sqrt(z*z + sigma*sigma))

def svi_minimum(a, b, rho, sigma):
    return a + b * sigma * np.sqrt(max(1.0 - rho*rho, 0.0))

def svi_from_unconstrained(x):
    # x = [log minimum variance, log b, atanh rho, m, log sigma]
    w_min = np.exp(x[0])
    b = np.exp(x[1])
    rho = np.tanh(x[2])
    m = x[3]
    sigma = np.exp(x[4])
    a = w_min - b * sigma * np.sqrt(max(1.0 - rho*rho, 1e-14))
    return a, b, rho, m, sigma

def unconstrained_from_svi(a, b, rho, m, sigma):
    w_min = max(svi_minimum(a, b, rho, sigma), 1e-8)
    return np.array([
        np.log(w_min),
        np.log(max(b, 1e-8)),
        np.arctanh(np.clip(rho, -0.995, 0.995)),
        m,
        np.log(max(sigma, 1e-6)),
    ])

def svi_derivatives(k, a, b, rho, m, sigma):
    z = np.asarray(k, float) - m
    root = np.sqrt(z*z + sigma*sigma)
    wk = b * (rho + z / root)
    wkk = b * sigma*sigma / np.maximum(root**3, 1e-18)
    return wk, wkk

def gatheral_g(k, w, wk, wkk):
    w = np.maximum(np.asarray(w, float), 1e-12)
    return (
        (1.0 - np.asarray(k) * wk / (2.0*w))**2
        - 0.25 * wk*wk * (1.0/w + 0.25)
        + 0.5 * wkk
    )

def fit_svi_slice(g, previous=None, progress_bar=None):
    g = g.sort_values("k")
    k = g["k"].to_numpy(float)
    w = g["w_market"].to_numpy(float)
    weight = g["fit_weight"].to_numpy(float)
    sqrt_weight = np.sqrt(weight)

    atm = float(np.interp(0.0, k, w))
    k_med = float(np.median(k))

    starts = []
    if previous is not None:
        starts.append(unconstrained_from_svi(
            previous["a"], previous["b"], previous["rho"],
            previous["m"], previous["sigma"]
        ))

    for rho0 in (-0.85, -0.60, -0.30):
        for sig0 in (0.05, 0.15, 0.30):
            b0 = max(0.02, 0.25 * np.nanstd(w) / max(np.nanstd(k), 1e-3))
            amin0 = max(0.5 * atm, 1e-6)
            a0 = amin0 - b0 * sig0 * np.sqrt(1-rho0*rho0)
            starts.append(unconstrained_from_svi(a0, b0, rho0, k_med, sig0))

    k_pen = np.linspace(K_MIN, K_MAX, 301)

    def residual(x):
        a, b, rho, m, sigma = svi_from_unconstrained(x)
        fitted = svi_raw_w(k, a, b, rho, m, sigma)
        data_resid = sqrt_weight * (fitted - w)

        # Soft Roger-Lee and butterfly penalties.
        left_slope = b * (1.0 - rho)
        right_slope = b * (1.0 + rho)
        slope_pen = np.array([
            max(left_slope - 1.98, 0.0),
            max(right_slope - 1.98, 0.0),
        ]) * 10.0

        wp = svi_raw_w(k_pen, a, b, rho, m, sigma)
        wkp, wkkp = svi_derivatives(k_pen, a, b, rho, m, sigma)
        gp = gatheral_g(k_pen, wp, wkp, wkkp)
        butterfly_pen = 20.0 * np.minimum(gp, 0.0)

        return np.concatenate([data_resid, slope_pen, butterfly_pen])

    if progress_bar is not None:
        progress_bar.set_postfix(
            date=str(pd.Timestamp(g["date"].iloc[0]).date()),
            expiry=str(pd.Timestamp(g["exdate"].iloc[0]).date()),
            refresh=False,
        )
    
    # Every completed optimiser start advances the overall bar by a
    # fraction of one slice. The total contribution of this slice is 1.
    progress_step = 1.0 / len(starts)


    best = None
    for x0 in starts:
        try:
            res = least_squares(
                residual,
                x0=x0,
                max_nfev=1500,
                xtol=1e-11,
                ftol=1e-11,
                gtol=1e-11,
            )
            score = float(np.sum(residual(res.x)**2))
            if best is None or score < best[0]:
                best = (score, res)
        except Exception:
            pass

        finally:
            if progress_bar is not None:
                progress_bar.update(progress_step)

    if best is None:
        raise RuntimeError("All SVI starts failed.")

    _, res = best
    a, b, rho, m, sigma = svi_from_unconstrained(res.x)
    w_fit = svi_raw_w(k, a, b, rho, m, sigma)
    return {
        "a": a, "b": b, "rho": rho, "m": m, "sigma": sigma,
        "n": len(g),
        "sse_w": float(np.sum(weight * (w_fit - w)**2)),
        "success": bool(res.success),
        "message": res.message,
    }

def calibrate_day_svi(day_otm, progress_bar=None):
    rows = []
    previous = None
    for (exdate, am_settlement), g in day_otm.groupby(
        ["exdate", "am_settlement"], sort=True
    ):
        try:
            fit = fit_svi_slice(g, previous=previous, progress_bar=progress_bar)
            fit.update({
                "date": g["date"].iloc[0],
                "exdate": exdate,
                "am_settlement": am_settlement,
                "T": float(g["T"].iloc[0]),
                "F": float(g["F"].iloc[0]),
                "D": float(g["D"].iloc[0]),
                "r": float(g["r"].iloc[0]),
                "carry": float(g["carry"].iloc[0]),
            })
            rows.append(fit)
            previous = fit
        except Exception as exc:
            tqdm.write(f"Skipping {exdate} AM={am_settlement}: {exc}")
    return pd.DataFrame(rows).sort_values("T").reset_index(drop=True)

## 10. Calibrate every daily SVI surface and compute same-day fitted prices

In [35]:
all_svi_params = []

date_groups = otm.groupby("date", sort=True)

total_slices = otm.groupby(["date", "exdate", "am_settlement"]).ngroups

with tqdm(
    total=total_slices,
    desc="Calibrating raw SVI slices",
    unit="slice",
    dynamic_ncols=True,
    mininterval=0.5,
) as progress_bar:

    for date, day_otm in date_groups:
        fitted = calibrate_day_svi(day_otm, progress_bar=progress_bar)

        if not fitted.empty:
            all_svi_params.append(fitted)

if not all_svi_params:
    raise RuntimeError("No SVI slices were fitted.")

svi_params = pd.concat(all_svi_params, ignore_index=True)

print("Total fitted SVI slices:", len(svi_params))

Calibrating raw SVI slices:   0%|          | 1.1000000000000003/809 [00:29<5:56:37, 26.49s/slice, date=2024-03-01, expiry=2024-03-19]


KeyboardInterrupt: 

In [ ]:
def add_same_day_svi_fit(otm_df, params_df):
    key = ["date", "exdate", "am_settlement"]
    merged = otm_df.merge(
        params_df[key + ["a", "b", "rho", "m", "sigma"]],
        on=key,
        how="inner",
        validate="many_to_one",
    )
    merged["w_svi"] = svi_raw_w(
        merged["k"], merged["a"], merged["b"], merged["rho"],
        merged["m"], merged["sigma"]
    )
    merged["iv_svi"] = np.sqrt(np.maximum(merged["w_svi"], 1e-12) / merged["T"])
    merged["price_svi_bs"] = bs_forward_price(
        merged["cp_otm"], merged["F"], merged["K"],
        merged["T"], merged["D"], merged["iv_svi"]
    )
    merged["pricing_error_svi"] = merged["price_svi_bs"] - merged["mid_otm"]
    merged["inside_spread_svi"] = (
        (merged["price_svi_bs"] >= merged["bid_otm"])
        & (merged["price_svi_bs"] <= merged["ask_otm"])
    )
    return merged

same_day_svi = add_same_day_svi_fit(otm, svi_params)

same_day_summary = pd.Series({
    "n": len(same_day_svi),
    "RMSE": np.sqrt(np.mean(same_day_svi["pricing_error_svi"]**2)),
    "MAE": np.mean(np.abs(same_day_svi["pricing_error_svi"])),
    "inside_bid_ask": np.mean(same_day_svi["inside_spread_svi"]),
})
same_day_summary

## 11. Build a smooth calendar-safe total-variance surface

The maturity interpolation is performed **directly on total variance**, not separately on the five SVI parameters.

For every common $k$-grid point:

1. evaluate all fitted SVI slices;
2. apply isotonic regression to $T\mapsto w(T,k)$;
3. construct a monotone PCHIP interpolant;
4. differentiate it analytically to obtain $w_T$.

This directly targets the calendar condition

$$
\partial_T w(k,T)\geq0.
$$

In [ ]:
class SmoothTotalVarianceSurface:
    def __init__(self, params_day, k_grid=None):
        self.params = params_day.sort_values("T").reset_index(drop=True).copy()
        self.T_nodes = self.params["T"].to_numpy(float)
        self.k_grid = (
            np.linspace(K_MIN, K_MAX, N_K)
            if k_grid is None else np.asarray(k_grid, float)
        )

        raw = np.vstack([
            svi_raw_w(
                self.k_grid, row.a, row.b, row.rho, row.m, row.sigma
            )
            for row in self.params.itertuples()
        ])

        ir = IsotonicRegression(increasing=True, out_of_bounds="clip")
        iso = np.empty_like(raw)
        for j in range(raw.shape[1]):
            iso[:, j] = ir.fit_transform(self.T_nodes, raw[:, j])

        # Enforce a tiny strict maturity increase before PCHIP.
        for i in range(1, len(self.T_nodes)):
            dt = self.T_nodes[i] - self.T_nodes[i-1]
            iso[i] = np.maximum(iso[i], iso[i-1] + DUPIRE_WT_FLOOR * dt)

        self.W_nodes = iso
        self._pchip_T = [
            PchipInterpolator(self.T_nodes, iso[:, j], extrapolate=False)
            for j in range(len(self.k_grid))
        ]

    def _evaluate_on_k_grid(self, T, derivative=0):
        T_arr = np.atleast_1d(np.asarray(T, float))
        T_clip = np.clip(T_arr, self.T_nodes[0], self.T_nodes[-1])
        out = np.empty((len(T_clip), len(self.k_grid)))
        for j, f in enumerate(self._pchip_T):
            out[:, j] = f.derivative(derivative)(T_clip)
        return out

    def w(self, T, k):
        T_arr, k_arr = np.broadcast_arrays(np.asarray(T, float), np.asarray(k, float))
        values = self._evaluate_on_k_grid(T_arr.ravel(), derivative=0)
        ans = np.array([
            np.interp(kk, self.k_grid, values[i])
            for i, kk in enumerate(k_arr.ravel())
        ])
        return ans.reshape(T_arr.shape)

    def wT(self, T, k):
        T_arr, k_arr = np.broadcast_arrays(np.asarray(T, float), np.asarray(k, float))
        values = self._evaluate_on_k_grid(T_arr.ravel(), derivative=1)
        ans = np.array([
            np.interp(kk, self.k_grid, values[i])
            for i, kk in enumerate(k_arr.ravel())
        ])
        return ans.reshape(T_arr.shape)

    def dense_grids(self, T_grid=None):
        if T_grid is None:
            T_grid = np.unique(np.concatenate([
                self.T_nodes,
                np.linspace(self.T_nodes[0], self.T_nodes[-1], max(80, 4*len(self.T_nodes)))
            ]))
        T_grid = np.asarray(T_grid, float)

        W = self._evaluate_on_k_grid(T_grid, derivative=0)
        WT = self._evaluate_on_k_grid(T_grid, derivative=1)
        WK = np.gradient(W, self.k_grid, axis=1, edge_order=2)
        WKK = np.gradient(WK, self.k_grid, axis=1, edge_order=2)
        return T_grid, self.k_grid, W, WT, WK, WKK

## 12. Dupire local-volatility extraction

In forward log-moneyness coordinates,

$$
\sigma_{\mathrm{loc}}^2(T,k)
=
\frac{w_T(T,k)}
{
\left(1-\frac{k w_k}{2w}\right)^2
-\frac{w_k^2}{4}\left(\frac1w+\frac14\right)
+\frac12w_{kk}
}.
$$

The denominator is the Gatheral butterfly-density function $g(T,k)$. The notebook records every floor or cap hit.

In [ ]:
class LocalVolSurface:
    def __init__(self, total_variance_surface):
        self.tv = total_variance_surface
        T_grid, k_grid, W, WT, WK, WKK = self.tv.dense_grids()

        K2 = k_grid[None, :]
        G = gatheral_g(K2, W, WK, WKK)

        WT_used = np.maximum(WT, DUPIRE_WT_FLOOR)
        G_used = np.maximum(G, DUPIRE_G_FLOOR)
        lv2 = WT_used / G_used
        sigma = np.sqrt(np.clip(
            lv2,
            LOCAL_VOL_MIN**2,
            LOCAL_VOL_MAX**2
        ))

        self.T_grid = T_grid
        self.k_grid = k_grid
        self.W = W
        self.WT = WT
        self.WK = WK
        self.WKK = WKK
        self.G = G
        self.sigma_grid = sigma

        self.interpolator = RegularGridInterpolator(
            (T_grid, k_grid),
            sigma,
            bounds_error=False,
            fill_value=None,
        )

        self.diagnostics = pd.Series({
            "min_w": np.nanmin(W),
            "min_wT_raw": np.nanmin(WT),
            "fraction_wT_floored": np.mean(WT < DUPIRE_WT_FLOOR),
            "min_g_raw": np.nanmin(G),
            "fraction_g_floored": np.mean(G < DUPIRE_G_FLOOR),
            "min_sigma": np.nanmin(sigma),
            "max_sigma": np.nanmax(sigma),
            "fraction_sigma_floor": np.mean(sigma <= LOCAL_VOL_MIN + 1e-12),
            "fraction_sigma_cap": np.mean(sigma >= LOCAL_VOL_MAX - 1e-12),
        })

    def sigma_tk(self, T, k):
        T_arr, k_arr = np.broadcast_arrays(np.asarray(T, float), np.asarray(k, float))
        pts = np.column_stack([
            np.clip(T_arr.ravel(), self.T_grid[0], self.T_grid[-1]),
            np.clip(k_arr.ravel(), self.k_grid[0], self.k_grid[-1]),
        ])
        return self.interpolator(pts).reshape(T_arr.shape)

    def sigma_tS(self, elapsed_t, S, S0, carry):
        # Local-vol surface is indexed by horizon from the calibration date.
        F_t = S0 * np.exp(carry * elapsed_t)
        k = np.log(np.maximum(S, 1e-300) / max(F_t, 1e-300))
        return self.sigma_tk(elapsed_t, k)

In [ ]:
dates = sorted(svi_params["date"].unique())
example_date = pd.Timestamp(EXAMPLE_DATE) if EXAMPLE_DATE is not None else pd.Timestamp(dates[0])

params_example = svi_params[svi_params["date"] == example_date].copy()
tv_example = SmoothTotalVarianceSurface(params_example)
lv_example = LocalVolSurface(tv_example)

print("Example date:", example_date.date())
display(lv_example.diagnostics.to_frame("value"))

## 13. Visual diagnostics for one calibration date

In [ ]:
day_otm = same_day_svi[same_day_svi["date"] == example_date]

fig, ax = plt.subplots(figsize=(10, 6))
for T, g in day_otm.groupby("T"):
    ax.scatter(g["k"], g["w_market"], s=10, alpha=0.45)
    order = np.argsort(g["k"].to_numpy())
    ax.plot(
        g["k"].to_numpy()[order],
        g["w_svi"].to_numpy()[order],
        linewidth=1.2,
        label=f"{int(round(T*365))}d",
    )
ax.set_title(f"Raw SVI total-variance slices — {example_date.date()}")
ax.set_xlabel("Forward log-moneyness k")
ax.set_ylabel("Total implied variance w")
ax.legend(ncol=3, fontsize=8)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
mesh = ax.pcolormesh(
    lv_example.k_grid,
    lv_example.T_grid * 365,
    lv_example.sigma_grid,
    shading="auto",
)
fig.colorbar(mesh, ax=ax, label="Local volatility")
ax.set_title(f"Dupire local-volatility surface — {example_date.date()}")
ax.set_xlabel("Forward log-moneyness k")
ax.set_ylabel("Maturity in days")
plt.show()

## 14. Crank--Nicolson PDE engine in log spot

For

$$
dS_t=c(t)S_t\,dt+\sigma_{\mathrm{loc}}(t,S_t)S_t\,dW_t,
$$

the value $V(t,x)$, with $x=\log(S/S_0)$, solves

$$
V_t+
\left(c-\frac12\sigma_{\mathrm{loc}}^2\right)V_x
+\frac12\sigma_{\mathrm{loc}}^2V_{xx}
-rV=0.
$$

A single PDE solve for an expiry returns the full value curve, allowing interpolation across all strikes and extraction of spot delta from $V_x/S$.

In [ ]:
def thomas_solve(a, b, c, d):
    a = np.asarray(a, float).copy()
    b = np.asarray(b, float).copy()
    c = np.asarray(c, float).copy()
    d = np.asarray(d, float).copy()

    for i in range(1, len(b)):
        m = a[i] / b[i-1]
        b[i] -= m * c[i-1]
        d[i] -= m * d[i-1]

    x = np.empty_like(d)
    x[-1] = d[-1] / b[-1]
    for i in range(len(b)-2, -1, -1):
        x[i] = (d[i] - c[i]*x[i+1]) / b[i]
    return x

def pde_price_curve(
    lv_surface,
    S0,
    K,
    T,
    r,
    carry,
    cp_flag,
    nx=PDE_NX,
    nt=None,
    x_width=PDE_X_WIDTH,
):
    K = float(K)
    T = float(T)
    if nt is None:
        nt = max(120, int(np.ceil(PDE_NT_PER_YEAR * T)))

    if nx % 2 == 0:
        nx += 1

    x = np.linspace(-x_width, x_width, nx)
    S = S0 * np.exp(x)
    dx = x[1] - x[0]
    dt = T / nt

    if cp_flag == "C":
        V = np.maximum(S - K, 0.0)
    else:
        V = np.maximum(K - S, 0.0)

    def boundaries(t_now, V):
        tau = max(T - t_now, 0.0)
        disc_r = np.exp(-r * tau)
        disc_q = np.exp(-(r-carry) * tau)
        if cp_flag == "C":
            V[0] = 0.0
            V[-1] = max(S[-1]*disc_q - K*disc_r, 0.0)
        else:
            V[-1] = 0.0
            V[0] = max(K*disc_r - S[0]*disc_q, 0.0)
        return V

    V = boundaries(T, V)

    inv_dx = 1.0 / dx
    inv_dx2 = inv_dx**2

    for n in range(nt):
        calendar_time = T - (n + 0.5) * dt
        elapsed = max(T - calendar_time, 0.0)

        sigma = lv_surface.sigma_tS(
            max(elapsed, lv_surface.T_grid[0]),
            S,
            S0,
            carry,
        )
        B = 0.5 * sigma**2
        A = carry - 0.5 * sigma**2

        lower = B*inv_dx2 - 0.5*A*inv_dx
        diag = -2.0*B*inv_dx2 - r
        upper = B*inv_dx2 + 0.5*A*inv_dx

        lo = lower[1:-1]
        di = diag[1:-1]
        up = upper[1:-1]

        aa = -0.5*dt*lo
        bb = 1.0 - 0.5*dt*di
        cc = -0.5*dt*up

        rhs = (
            (1.0 + 0.5*dt*di)*V[1:-1]
            + 0.5*dt*up*V[2:]
            + 0.5*dt*lo*V[:-2]
        )

        V_old = V.copy()
        V = boundaries(T - (n+1)*dt, V)
        rhs[0] -= aa[0]*V[0]
        rhs[-1] -= cc[-1]*V[-1]

        aa[0] = 0.0
        cc[-1] = 0.0
        V[1:-1] = thomas_solve(aa, bb, cc, rhs)

    Vx = np.gradient(V, dx, edge_order=2)
    delta = Vx / np.maximum(S, 1e-300)
    return S, V, delta

def pde_price_delta_one(
    lv_surface, S0, K, T, r, carry, cp_flag,
    nx=PDE_NX, nt=None, x_width=PDE_X_WIDTH,
):
    S_grid, V_grid, delta_grid = pde_price_curve(
        lv_surface, S0, K, T, r, carry, cp_flag,
        nx=nx, nt=nt, x_width=x_width
    )
    price = float(np.interp(S0, S_grid, V_grid))
    delta = float(np.interp(S0, S_grid, delta_grid))
    return price, delta

## 15. Constant-volatility validation of the PDE solver

In [ ]:
class ConstantVol:
    def __init__(self, sigma):
        self.sigma = float(sigma)
        self.T_grid = np.array([1e-6, 5.0])
    def sigma_tS(self, elapsed_t, S, S0, carry):
        return np.full_like(np.asarray(S, float), self.sigma)

test_S0, test_K, test_T = 5000.0, 5100.0, 60/365
test_r, test_carry, test_sigma = 0.05, 0.04, 0.20
test_F = test_S0*np.exp(test_carry*test_T)
test_D = np.exp(-test_r*test_T)

pde_p, pde_d = pde_price_delta_one(
    ConstantVol(test_sigma),
    test_S0, test_K, test_T, test_r, test_carry, "C",
    nx=301, nt=300,
)
bs_p = float(bs_forward_price(
    "C", test_F, test_K, test_T, test_D, test_sigma
))

print("BS price:", bs_p)
print("PDE price:", pde_p)
print("absolute difference:", abs(pde_p-bs_p))
print("PDE delta:", pde_d)

## 16. Same-day LV-PDE recovery

This is a crucial separate test. The SVI fit can price well while its Dupire derivatives are too noisy for the LV model to recover the same surface.

By default, the notebook evaluates one representative contract per expiry. Set `RUN_FULL_SAME_DAY_PDE=True` for all OTM quotes.

In [ ]:
def build_lv_for_date(date):
    p = svi_params[svi_params["date"] == pd.Timestamp(date)].copy()
    if len(p) < 2:
        raise ValueError("At least two maturities are required for Dupire.")
    tv = SmoothTotalVarianceSurface(p)
    return LocalVolSurface(tv)

def same_day_pde_test(date, full=False):
    date = pd.Timestamp(date)
    lv = build_lv_for_date(date)
    day = otm[otm["date"] == date].copy()

    if not full:
        # Pick the quote closest to ATM for each expiry.
        day = (
            day.assign(abs_k=day["k"].abs())
            .sort_values("abs_k")
            .groupby(["exdate", "am_settlement"], as_index=False)
            .head(1)
        )

    rows = []

    iterator = tqdm(
        day.itertuples(index=False),
        total=len(day),
        desc=f"Same-day PDE {date.date()}",
        unit="option",
        dynamic_ncols=True,
    )

    for row in iterator:
        try:
            price, delta = pde_price_delta_one(
                lv,
                row.S,
                row.K,
                row.T,
                row.r,
                row.carry,
                row.cp_otm,
            )

            rows.append({
                "date": date,
                "exdate": row.exdate,
                "K": row.K,
                "cp_flag": row.cp_otm,
                "market_mid": row.mid_otm,
                "market_bid": row.bid_otm,
                "market_ask": row.ask_otm,
                "pde_price": price,
                "pde_delta": delta,
            })

        except Exception as exc:
            tqdm.write(f"PDE failure: {row.exdate} {row.K}: {exc}")

    result = pd.DataFrame(rows)
    if len(result):
        result["error"] = result["pde_price"] - result["market_mid"]
        result["inside_spread"] = (
            (result["pde_price"] >= result["market_bid"])
            & (result["pde_price"] <= result["market_ask"])
        )
    return result, lv

same_day_pde_example, lv_example_check = same_day_pde_test(
    example_date, full=RUN_FULL_SAME_DAY_PDE
)
display(same_day_pde_example.head())
if len(same_day_pde_example):
    print("Same-day LV-PDE MAE:",
          same_day_pde_example["error"].abs().mean())
    print("Inside spread:",
          same_day_pde_example["inside_spread"].mean())

## 17. Build identical-contract next-trading-day pairs

The day-$t$ local-volatility surface is frozen. The same option contract is then repriced on the next observed trading day with:

- the new spot;
- one fewer trading interval of remaining maturity;
- the next day's discount and forward curves;
- the **day-$t$ local-volatility function** unchanged.

The one-step self-financing hedge error is

$$
\varepsilon_{t,t+1}
=
V_{t+1}^{\mathrm{mkt}}-V_t^{\mathrm{mkt}}
-\Delta_t(S_{t+1}-S_t)
-r_t\left(V_t^{\mathrm{mkt}}-\Delta_tS_t\right)\Delta t.
$$

In [ ]:
def build_evaluation_pairs(quotes):
    trading_dates = sorted(quotes["date"].unique())
    next_date = {
        trading_dates[i]: trading_dates[i+1]
        for i in range(len(trading_dates)-1)
    }

    left = quotes.copy()
    left["next_date"] = left["date"].map(next_date)

    right_cols = [
        "date", "optionid", "cp_flag", "K", "exdate", "am_settlement",
        "mid", "bid", "ask", "S", "F", "D", "r", "carry", "T"
    ]
    right = quotes[right_cols].rename(columns={
        "date": "next_date",
        "mid": "mid_next",
        "bid": "bid_next",
        "ask": "ask_next",
        "S": "S_next",
        "F": "F_next",
        "D": "D_next",
        "r": "r_next",
        "carry": "carry_next",
        "T": "T_next",
    })

    pairs = left.merge(
        right,
        on=["next_date", "optionid", "cp_flag", "K", "exdate", "am_settlement"],
        how="inner",
        validate="many_to_one",
    )
    return pairs

evaluation_pairs = build_evaluation_pairs(quotes_all)
print("Identical-contract one-day pairs:", len(evaluation_pairs))

## 18. Rolling frozen-LV pricing and hedging backtest

This section is disabled by default because a full all-contract Crank--Nicolson sweep is computationally expensive.

The implementation prices each contract individually for clarity. For the final production notebook, this can be accelerated by solving one PDE per expiry and interpolating across all strikes.

In [ ]:
def rolling_lv_backtest(max_transitions=None):
    dates = sorted(svi_params["date"].unique())

    transitions = list(zip(dates[:-1], dates[1:]))

    if max_transitions is not None:
        transitions = transitions[:max_transitions]

    all_rows = []

    transition_iterator = tqdm(
        transitions,
        total=len(transitions),
        desc="Rolling LV transitions",
        unit="transition",
        dynamic_ncols=True,
    )

    for d0, d1 in transition_iterator:
        d0 = pd.Timestamp(d0)
        d1 = pd.Timestamp(d1)

        transition_iterator.set_postfix(
            transition=(f"{d0.date()}->{d1.date()}"), rows=len(all_rows), refresh=False)

        try:
            lv = build_lv_for_date(d0)

        except Exception as exc:
            tqdm.write(f"Skipping {d0.date()}: {exc}")
            continue

        pairs = evaluation_pairs[
            (evaluation_pairs["date"] == d0)
            & (evaluation_pairs["next_date"]== d1)
        ].copy()

        Tmin = lv.T_grid[0]
        Tmax = lv.T_grid[-1]

        pairs = pairs[pairs["T"].between(Tmin, Tmax)].copy()

        pair_iterator = tqdm(
            pairs.itertuples(index=False),
            total=len(pairs),
            desc=(f"Options {d0.date()}->{d1.date()}"),
            unit="option",
            leave=False,
            dynamic_ncols=True,
        )

        for row in pair_iterator:
            try:
                price_t, delta_t = (
                    pde_price_delta_one(
                        lv,
                        row.S,
                        row.K,
                        row.T,
                        row.r,
                        row.carry,
                        row.cp_flag,
                    )
                )

                price_next, _ = (
                    pde_price_delta_one(
                        lv,
                        row.S_next,
                        row.K,
                        row.T_next,
                        row.r_next,
                        row.carry_next,
                        row.cp_flag,
                    )
                )

                dt = (d1 - d0).days / 365.0

                cash_t = (row.mid - delta_t * row.S)

                hedge_error = (
                    row.mid_next
                    - row.mid
                    - delta_t * (row.S_next - row.S) 
                    - row.r * cash_t * dt
                )

                all_rows.append({
                    "date": d0,
                    "next_date": d1,
                    "optionid": row.optionid,
                    "cp_flag": row.cp_flag,
                    "K": row.K,
                    "exdate": row.exdate,
                    "T": row.T,
                    "k": row.k,
                    "mid": row.mid,
                    "mid_next": row.mid_next,
                    "model_price_t": price_t,
                    "model_price_next": price_next,
                    "delta_t": delta_t,
                    "next_day_pricing_error": (price_next - row.mid_next),
                    "hedging_error": (hedge_error),
                    "unhedged_change": (row.mid_next - row.mid),
                })

            except Exception:
                continue

    return pd.DataFrame(all_rows)

if RUN_ROLLING_BACKTEST:
    rolling_results = rolling_lv_backtest(MAX_ROLLING_TRANSITIONS)
else:
    rolling_results = pd.DataFrame()
    print("Rolling backtest disabled. Set RUN_ROLLING_BACKTEST=True and rerun.")

## 19. Summary metrics

In [ ]:
def metric_summary(df):
    if df.empty:
        return pd.Series(dtype=float)

    return pd.Series({
        "n": len(df),
        "next_day_RMSE": np.sqrt(np.mean(df["next_day_pricing_error"]**2)),
        "next_day_MAE": np.mean(np.abs(df["next_day_pricing_error"])),
        "hedging_RMSE": np.sqrt(np.mean(df["hedging_error"]**2)),
        "hedging_MAE": np.mean(np.abs(df["hedging_error"])),
        "hedging_bias": np.mean(df["hedging_error"]),
        "unhedged_MAE": np.mean(np.abs(df["unhedged_change"])),
        "hedging_improvement": (
            1.0
            - np.mean(np.abs(df["hedging_error"]))
            / np.mean(np.abs(df["unhedged_change"]))
        ),
    })

display(same_day_summary.to_frame("same_day_SVI"))
if not rolling_results.empty:
    display(metric_summary(rolling_results).to_frame("rolling_LV"))

## 20. Diagnostics by maturity and moneyness

These tables should be used before interpreting any aggregate result. Local volatility can behave very differently for short maturities and far-wing contracts.

In [ ]:
if not rolling_results.empty:
    rolling_results["dte_bucket"] = pd.cut(
        rolling_results["T"]*365,
        bins=[0, 30, 60, 90, 180, 365, np.inf],
        right=True,
    )
    rolling_results["abs_k_bucket"] = pd.cut(
        rolling_results["k"].abs(),
        bins=[0, 0.025, 0.05, 0.10, 0.20, 0.30, np.inf],
        right=True,
    )

    display(
        rolling_results.groupby("dte_bucket", observed=True).apply(metric_summary)
    )
    display(
        rolling_results.groupby("abs_k_bucket", observed=True).apply(metric_summary)
    )

## 21. Final modelling notes

### What is fixed in this version

- The cleaning is the **preliminary LV cleaning**, not the unified ad hoc-BS benchmark cleaning.
- Raw SVI is the final cross-sectional representation.
- Total variance is repaired and interpolated directly in maturity.
- Dupire inversion is performed only after explicit calendar and butterfly diagnostics.
- The hedge is the PDE spot delta from the LV model.

### What remains to be finalised

1. Run the complete rolling backtest with production PDE resolution.
2. Compare alternative weighting:
   equal, inverse spread, inverse variance and robust Huber weighting.
3. Compare strict rejection against soft penalisation for negative $g$.
4. Decide whether the final benchmark notebook should:
   - fit on this strict LV subset and evaluate on the shared BS universe; or
   - use exactly the same calibration universe as the ad hoc BS model.
5. Vectorise the PDE by expiry for the final high-speed implementation.